# Feature extractors: basics

This tutorial introduces the `light_curve` feature extractor interface:
creating features, combining them with [`Extractor`](../api/meta/#light_curve.Extractor),
reading names and descriptions, and batch processing.

## Setup

We create a synthetic sinusoidal light curve with Gaussian noise:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(42)
t = np.sort(rng.uniform(0, 100, 200))
m = 15.0 + 0.3 * np.sin(2 * np.pi * t / 20) + rng.normal(0, 0.05, 200)
err = np.full(200, 0.05)

print(f'Observations: {len(t)}')
print(f'Time range:   {t[0]:.1f} – {t[-1]:.1f} days')
print(f'Mag range:    {m.min():.2f} – {m.max():.2f}')

## Single feature

Each feature class is callable. It accepts `(t, m, sigma)` arrays and returns a NumPy array.
The `.names` attribute lists the output column names.

Here we use [`Amplitude`](../api/statistical/#light_curve.Amplitude) — the half peak-to-peak amplitude:

In [ ]:
amp = licu.Amplitude()
result = amp(t, m, err)
print(f'names:  {amp.names}')
print(f'result: {result}')

`.descriptions` gives a human-readable explanation of each output:

In [ ]:
print(amp.descriptions)

## Combining features with `Extractor`

[`Extractor`](../api/meta/#light_curve.Extractor) combines multiple features into a single callable evaluated in one pass.
It is especially efficient for **cheap features** (statistical moments, variability
indices, etc.) because it avoids repeated passes over the data and reduces Python–Rust
call overhead:

In [ ]:
ext = licu.Extractor(
    licu.Amplitude(),
    licu.BeyondNStd(nstd=1),
    licu.BeyondNStd(nstd=2),
    licu.StandardDeviation(),
    licu.WeightedMean(),
    licu.LinearFit(),
    licu.StetsonK(),
)
result = ext(t, m, err)
for name, value in zip(ext.names, result):
    print(f'  {name:35s} = {value:.5f}')

## Batch processing with `.many()`

`.many()` processes a list of `(t, m, sigma)` tuples and returns a 2-D NumPy array
(shape `(N, n_features)`). It supports **multi-threading** (enabled by default via the
`n_jobs` parameter) and is the preferred path for large datasets:

In [ ]:
n_lc = 5_000
light_curves = []
for _ in range(n_lc):
    n = int(rng.integers(50, 200))
    t_i = np.sort(rng.uniform(0, 100, n))
    m_i = 15.0 + rng.normal(0, 0.2, n)
    light_curves.append((t_i, m_i, np.full(n, 0.05)))

# Batch extraction — one call for all light curves, multi-threaded
results = licu.Amplitude().many(light_curves)
print(f'Extracted from {n_lc} light curves: shape = {results.shape}')
print(f'Mean amplitude = {results.mean():.4f} mag')

## Batch processing with columnar libraries

`.many()` accepts any Arrow-compatible array — **PyArrow**, **Polars Series**, or
**nested-pandas** — as well as plain lists of `(t, m, sigma)` tuples.
All inputs must be in **`List<Struct<t, m, sigma>>`** format (one list entry per light curve).

In [ ]:
import pyarrow as pa
import polars as pl
import nested_pandas as npd
import pandas as pd

# ── helpers ────────────────────────────────────────────────────────────────
def make_struct_array(rng):
    """Build a single-LC StructArray with fields t, m, sigma."""
    n = int(rng.integers(80, 150))
    t_i = np.sort(rng.uniform(0, 100, n))
    m_i = 15.0 + rng.normal(0, 0.3, n)
    e_i = np.full(n, 0.05)
    return pa.StructArray.from_arrays(
        [pa.array(t_i), pa.array(m_i), pa.array(e_i)],
        names=["t", "m", "sigma"],
    )

structs = [make_struct_array(rng) for _ in range(4)]
lengths = [len(s) for s in structs]
offsets = pa.array([0] + [sum(lengths[:i+1]) for i in range(len(lengths))])
arrow_lcs = pa.ListArray.from_arrays(offsets, pa.concat_arrays(structs))

# ── PyArrow ────────────────────────────────────────────────────────────────
amp_arrow = licu.Amplitude().many(arrow_lcs, arrow_fields=["t", "m", "sigma"])
print("PyArrow  amplitudes:", amp_arrow.flatten())

# ── Polars Series (wraps the same Arrow memory) ────────────────────────────
polars_lcs = pl.Series(arrow_lcs)
amp_polars = licu.Amplitude().many(polars_lcs, arrow_fields=["t", "m", "sigma"])
print("Polars   amplitudes:", amp_polars.flatten())

# ── nested-pandas NestedFrame ──────────────────────────────────────────────
flat_t   = np.concatenate([np.asarray(s.field('t'))     for s in structs])
flat_m   = np.concatenate([np.asarray(s.field('m'))     for s in structs])
flat_sig = np.concatenate([np.asarray(s.field('sigma')) for s in structs])
obj_ids  = np.repeat(np.arange(len(structs)), lengths)

nf = npd.NestedFrame.from_flat(
    pd.DataFrame({'t': flat_t, 'm': flat_m, 'sigma': flat_sig, 'obj_id': obj_ids}),
    base_columns=['obj_id'],
    nested_columns=['t', 'm', 'sigma'],
    on='obj_id',
    name='lc',
)
# .array.list_array exposes the underlying Arrow ChunkedArray
amp_npd = licu.Amplitude().many(nf['lc'].array.list_array, arrow_fields=['t', 'm', 'sigma'])
print("nested-pandas amplitudes:", amp_npd.flatten())

## Next steps

- [Feature table](../) — all 40+ extractors grouped by category
- [API reference](../api/) — full signatures and equations
- [Periodogram tutorial](../periodogram/) — Lomb–Scargle and period search